In [ ]:
# ── Cell 0: Locate datasets, output scaffold ──────────────────────────────────
from pathlib import Path
import json, shutil, os, sys, re
import numpy as np, torch, torch.nn as nn, pandas as pd
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.metrics import f1_score, confusion_matrix, classification_report

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
print(f"Device: {DEVICE}")

def find_dir(*fragments):
    for frag in fragments:
        m = [p for p in Path("/kaggle/input").rglob("*") if p.is_dir() and frag in p.name.lower()]
        if m: return m[0]
    return None

def find_file(*fragments):
    for frag in fragments:
        m = [p for p in Path("/kaggle/input").rglob("*") if p.is_file() and frag in p.name.lower()]
        if m: return m[0]
    return None

KAN_QWEN = find_dir("kan_qwen", "kan-qwen")
KAN_LLAMA = find_dir("kan_llama", "llama-kan")
PREV_V3 = find_dir("pre_run_v3", "pre-run_v3", "v3_final")
KNOWN_IMG_DIR = find_dir("known")
NEW_IMG_DIR = find_dir("new")
REVIEW_B = find_file("review_b_ambiguous_queries")

for name, p in [("KAN_QWEN",KAN_QWEN),("KAN_LLAMA",KAN_LLAMA),("PREV_V3",PREV_V3),
                ("KNOWN_IMG_DIR",KNOWN_IMG_DIR),("NEW_IMG_DIR",NEW_IMG_DIR),("REVIEW_B",REVIEW_B)]:
    print(f"{'✅' if p else '❌'} {name}: {p}")

OUT = Path("/kaggle/working/outputs")
for sub in ["class_mappings","splits","canonical","human_benchmark","sanity_checks",
            "leakage_audit","pixel_clip_provenance","validation","part4_eval_prep"]:
    (OUT/sub).mkdir(parents=True, exist_ok=True)
print("✅ scaffold ready")

if not Path("/kaggle/working/fast-kan").exists():
    os.system("git clone https://github.com/ZiyaoLi/fast-kan.git /kaggle/working/fast-kan")
sys.path.append("/kaggle/working/fast-kan")

In [ ]:
# ── Check if the v3 mapping files are actually inside the pre-run-v4 dataset ─
pre_run_v4_dir = find_dir("pre-run-v4", "pre_run_v4")
print(f"pre-run-v4 location: {pre_run_v4_dir}")

if pre_run_v4_dir:
    for pattern in ["*class_mapping*.json", "*35way*.pt", "*shared29*.pt", "CANONICAL_RESULTS.md"]:
        matches = list(pre_run_v4_dir.rglob(pattern))
        print(f"\n'{pattern}': {len(matches)} found")
        for m in matches[:5]:
            print(f"  {m}")

In [ ]:
# ── Check what's actually inside pre-run-v4-1 ─────────────────────────────────
from pathlib import Path

candidate = Path("/kaggle/input/datasets/deboparna9das/pre-run-v4-1/kaggle/working")
print(f"Exists: {candidate.exists()}")

if candidate.exists():
    print("\nFull contents:")
    for p in sorted(candidate.rglob("*")):
        if p.is_file():
            print(f"  {p.relative_to(candidate)}")

    print("\n--- Checking for the specific files we need ---")
    for pattern in ["*class_mapping*.json", "*35way*.pt", "kan_qwen_35way_best.pt",
                     "mlp_qwen_35way_best.pt", "kan_llama_35way_best.pt", "mlp_llama_35way_best.pt"]:
        matches = list(candidate.rglob(pattern))
        print(f"'{pattern}': {len(matches)} found")
        for m in matches:
            print(f"    {m}")
else:
    print("Path doesn't exist — check the exact dataset slug and folder structure")
    print("\nTrying broader search instead:")
    base = Path("/kaggle/input/datasets/deboparna9das/pre-run-v4-1")
    if base.exists():
        for p in sorted(base.rglob("*"))[:50]:
            print(f"  {p}")

In [ ]:
# ── Fix: PREV_V3 points to pre-run-v4-1, not pre-run-v4 ──────────────────────
PREV_V3 = Path("/kaggle/input/datasets/deboparna9das/pre-run-v4-1/kaggle/working")
print(f"PREV_V3 set to: {PREV_V3}")
print(f"Exists: {PREV_V3.exists()}")

# Quick confirm all needed files resolve correctly
for pattern in ["qwen_35_class_mapping.json", "llama_35_class_mapping.json", "shared_29_class_mapping.json",
                "kan_qwen_35way_best.pt", "mlp_qwen_35way_best.pt", "kan_llama_35way_best.pt", "mlp_llama_35way_best.pt"]:
    found = list(PREV_V3.rglob(pattern))
    print(f"{'✅' if found else '❌'} {pattern}")

In [ ]:
# ── Cell 1: Shared architecture + helpers ─────────────────────────────────────
from fastkan import FastKAN

class KANRoute(nn.Module):
    def __init__(self, input_dim=896, proj_dim=48, hidden_dims=[48], num_tools=35, num_grids=4):
        super().__init__()
        self.proj = nn.Linear(input_dim, proj_dim)
        self.norm = nn.LayerNorm(proj_dim)
        self.kan  = FastKAN(layers_hidden=[proj_dim]+hidden_dims+[num_tools], num_grids=num_grids)
    def forward(self, x): return self.kan(self.norm(self.proj(x.float())))
    def count_parameters(self): return sum(p.numel() for p in self.parameters() if p.requires_grad)

class MLPBaseline(nn.Module):
    def __init__(self, input_dim=896, proj_dim=96, hidden_dims=[64,32], num_tools=35, dropout=0.3):
        super().__init__()
        layers = [nn.Linear(input_dim, proj_dim), nn.LayerNorm(proj_dim), nn.GELU(), nn.Dropout(dropout)]
        prev = proj_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.GELU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, num_tools))
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x.float())
    def count_parameters(self): return sum(p.numel() for p in self.parameters() if p.requires_grad)

class ToolRoutingDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32); self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

def evaluate_full(model, loader):
    model.eval(); preds, labels = [], []
    with torch.no_grad():
        for X_b,y_b in loader:
            X_b,y_b = X_b.float().to(DEVICE), y_b.to(DEVICE)
            p = model(X_b).argmax(-1)
            preds.extend(p.cpu().numpy()); labels.extend(y_b.cpu().numpy())
    return np.array(preds), np.array(labels)

def train_model(model, tr, va, name, epochs=60, lr=1e-3, wd=1e-4, is_kan=False, save_dir=None):
    if is_kan: lr, epochs = 5e-4, 80
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, steps_per_epoch=len(tr), epochs=epochs, pct_start=0.1)
    crit = nn.CrossEntropyLoss(label_smoothing=0.05)
    best, path = 0.0, (save_dir or OUT)/f"{name}_best.pt"
    for ep in range(1, epochs+1):
        model.train()
        for X_b,y_b in tr:
            X_b,y_b = X_b.float().to(DEVICE), y_b.to(DEVICE)
            opt.zero_grad(); loss = crit(model(X_b), y_b); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sch.step()
        model.eval(); c,t=0,0
        with torch.no_grad():
            for X_b,y_b in va:
                X_b,y_b = X_b.float().to(DEVICE), y_b.to(DEVICE)
                c += (model(X_b).argmax(-1)==y_b).sum().item(); t += len(y_b)
        acc = c/t
        if acc > best: best = acc; torch.save(model.state_dict(), path)
        if ep%20==0 or ep==1: print(f"  [{name}] ep{ep}/{epochs} val={acc:.4f}")
    model.load_state_dict(torch.load(path))
    return best
print("✅ ready")

In [ ]:
# ── Cell 2: Load raw datasets + class mappings from previous session ─────────
with open(KAN_QWEN / "augmented_dataset_final.json") as f:
    qwen_data = json.load(f)
qwen_text = np.load(KAN_QWEN / "aug_text_embeddings.npy")
qc = np.load(KAN_QWEN / "clip_features.npz", allow_pickle=True)
qwen_clip = {int(k):v for k,v in qc[qc.files[0]].item().items()}
qwen_clip_vecs = np.stack([qwen_clip[int(r["img_id"])] for r in qwen_data])
X_qwen_896 = np.concatenate([qwen_text, qwen_clip_vecs], axis=1).astype(np.float32)

with open(KAN_LLAMA / "raw_dataset_FINAL.json") as f:
    llama_data = json.load(f)
llama_text = np.load(KAN_LLAMA / "text_embeddings.npy")
lc = np.load(KAN_LLAMA / "clip_features.npz", allow_pickle=True)
llama_clip = {int(k):v for k,v in lc[lc.files[0]].item().items()}
llama_clip_vecs = np.stack([llama_clip[int(r["img_id"])] for r in llama_data])
X_llama_896 = np.concatenate([llama_text, llama_clip_vecs], axis=1).astype(np.float32)

with open(next(PREV_V3.rglob("qwen_35_class_mapping.json"))) as f:
    qwen_mapping = json.load(f)
with open(next(PREV_V3.rglob("llama_35_class_mapping.json"))) as f:
    llama_mapping = json.load(f)
with open(next(PREV_V3.rglob("shared_29_class_mapping.json"))) as f:
    shared_mapping = json.load(f)

qwen_classes = sorted(qwen_mapping["class_to_local_id"].keys())
llama_classes = sorted(llama_mapping["class_to_local_id"].keys())
shared_classes = sorted(shared_mapping["class_to_local_id"].keys())

y_qwen_local = np.array([qwen_mapping["class_to_local_id"][r["tool"]] for r in qwen_data], dtype=np.int64)
y_llama_local = np.array([llama_mapping["class_to_local_id"][r["tool"]] for r in llama_data], dtype=np.int64)

print(f"Qwen: {X_qwen_896.shape}, 35 classes | Llama: {X_llama_896.shape}, 35 classes | Shared: {len(shared_classes)}")

In [ ]:
# ── Cell 3: Qwen-35 full class distribution audit ─────────────────────────────
qwen_class_counts = []
for tool_name in qwen_classes:
    rows = [r for r in qwen_data if r["tool"] == tool_name]
    total_samples = len(rows)
    unique_queries = len(set(r["query"] for r in rows))
    unique_images = len(set(r["img_id"] for r in rows))
    qwen_class_counts.append(dict(
        tool_name=tool_name, total_samples=total_samples,
        unique_queries=unique_queries, unique_images=unique_images
    ))

qwen_dist_df = pd.DataFrame(qwen_class_counts).sort_values("total_samples")
qwen_dist_df.to_csv(OUT/"class_mappings"/"qwen35_class_distribution_before.csv", index=False)
print(qwen_dist_df.to_string(index=False))

below_100 = qwen_dist_df[qwen_dist_df["total_samples"] < 100]
print(f"\nClasses below 100 samples: {len(below_100)}")
print(below_100[["tool_name","total_samples"]].to_string(index=False))

In [ ]:
# ── Cell 4: Load Qwen2.5-7B, generate new queries for starved classes ────────
from transformers import AutoTokenizer, AutoModelForCausalLM
import random

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
print(f"Loading {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
teacher = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="auto",
    low_cpu_mem_usage=True, trust_remote_code=True,
)
teacher.eval()
print(f"✅ Loaded. VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")

TOOL_DESCRIPTIONS = {
    "image_colorization": "add realistic color to a grayscale or black-and-white image",
    "matting": "extract a foreground subject with soft, fine-detailed edges (hair, fur, transparency)",
    "pose_estimation": "determine body posture and joint/skeleton positions of a person",
}

def build_targeted_prompt(objects, tool_name):
    obj_str = ", ".join(objects) if objects else "various objects"
    messages = [
        {"role": "system", "content": (
            "You are a precise JSON generator for an AI benchmark dataset. "
            "You ALWAYS output valid JSON and nothing else. "
            "No markdown, no explanation, no code blocks."
        )},
        {"role": "user", "content": (
            f"Image contains: {obj_str}\n\n"
            f"Tool: {tool_name} — {TOOL_DESCRIPTIONS[tool_name]}\n\n"
            "Write ONE realistic, natural user query that would require this exact tool. "
            "Vary the phrasing style each time (direct request, casual question, or descriptive request).\n\n"
            'Output ONLY this JSON: {"query": "..."}'
        )}
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def generate_for_class(tool_name, target_count, existing_records, batch_size=8, max_new_tokens=100):
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    source_pool = [(r["img_id"], r.get("objects", [])) for r in existing_records]
    random.seed(SEED)
    random.shuffle(source_pool)

    new_records = []
    attempts = 0
    while len(new_records) < target_count and attempts < target_count * 3:
        start_idx = attempts % len(source_pool)
        batch = source_pool[start_idx: start_idx + batch_size]
        if not batch:
            attempts += batch_size; continue
        prompts = [build_targeted_prompt(objs, tool_name) for _, objs in batch]
        inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True, max_length=512).to(DEVICE)
        with torch.no_grad():
            output_ids = teacher.generate(
                **inputs, max_new_tokens=max_new_tokens, do_sample=True,
                temperature=0.8, top_p=0.9, repetition_penalty=1.15,
                pad_token_id=tokenizer.pad_token_id,
            )
        input_len = inputs["input_ids"].shape[1]
        for (img_id, objs), out_ids in zip(batch, output_ids):
            raw = tokenizer.decode(out_ids[input_len:], skip_special_tokens=True).strip()
            try:
                if "```" in raw:
                    raw = raw.split("```")[1].replace("json","",1).strip()
                parsed = json.loads(raw)
                query_text = parsed.get("query", "").strip()
                if query_text and len(new_records) < target_count:
                    new_records.append({
                        "img_id": img_id, "query": query_text, "tool": tool_name,
                        "objects": objs, "source": "item1_correction_qwen25_7b",
                    })
            except Exception:
                continue
        attempts += batch_size
        if attempts % 40 == 0:
            print(f"  [{tool_name}] {len(new_records)}/{target_count} generated...")
    return new_records

STARVED_CLASSES = below_100["tool_name"].tolist()
TARGET_PER_CLASS = 150

all_new_records = []
for tool_name in STARVED_CLASSES:
    current_n = int(below_100[below_100.tool_name==tool_name].total_samples.iloc[0])
    print(f"\nGenerating for '{tool_name}' (current: {current_n})...")
    new = generate_for_class(tool_name, TARGET_PER_CLASS, qwen_data)
    all_new_records.extend(new)
    print(f"  ✅ Generated {len(new)} new samples for {tool_name}")

print(f"\n✅ Total new records generated: {len(all_new_records)}")

In [ ]:
# ── Cell 4b: Extend TOOL_DESCRIPTIONS to cover ALL classes in below_100 ───────
print("Full list of classes below 100:")
print(below_100[["tool_name","total_samples"]].to_string(index=False))

# Generic fallback description generator for any class not already covered
GENERIC_DESCRIPTIONS = {
    "face_detection": "locate and identify human faces in an image",
    "text_detection_ocr": "detect and read visible text in an image",
    "logo_detection": "identify brand logos visible in an image",
    "vehicle_detection": "locate vehicles such as cars, trucks, or motorcycles in an image",
    "animal_detection": "locate animals present in an image",
    "gaze_estimation": "determine where a person is looking in an image",
    "age_gender_estimation": "estimate the age and gender of people in an image",
    "facial_expression_recognition": "identify the emotional expression on a person's face",
    "super_resolution": "increase the resolution and sharpness of an image",
    "image_deblurring": "remove blur and sharpen a blurry image",
    "image_denoising": "remove visual noise or grain from an image",
    "anomaly_detection": "identify something unusual or out of place in an image",
    "point_cloud_generation": "generate 3D point-cloud data from an image",
    # add more as needed based on what below_100 actually shows
}

for tool_name in below_100["tool_name"].tolist():
    if tool_name not in TOOL_DESCRIPTIONS:
        if tool_name in GENERIC_DESCRIPTIONS:
            TOOL_DESCRIPTIONS[tool_name] = GENERIC_DESCRIPTIONS[tool_name]
        else:
            # Last-resort fallback: derive a readable description from the tool name itself
            TOOL_DESCRIPTIONS[tool_name] = tool_name.replace("_", " ")
            print(f"⚠️ No hand-written description for '{tool_name}' — using auto-derived fallback: "
                  f"'{TOOL_DESCRIPTIONS[tool_name]}'. Consider writing a better one before continuing.")

print(f"\nTOOL_DESCRIPTIONS now covers: {sorted(TOOL_DESCRIPTIONS.keys())}")

In [ ]:
# ── Cell 4c: Resume generation for remaining classes only ────────────────────
already_done = {"image_colorization", "matting", "pose_estimation"}
remaining_classes = [t for t in below_100["tool_name"].tolist() if t not in already_done]
print(f"Remaining classes to generate: {remaining_classes}")

for tool_name in remaining_classes:
    current_n = int(below_100[below_100.tool_name==tool_name].total_samples.iloc[0])
    print(f"\nGenerating for '{tool_name}' (current: {current_n})...")
    new = generate_for_class(tool_name, TARGET_PER_CLASS, qwen_data)
    all_new_records.extend(new)
    print(f"  ✅ Generated {len(new)} new samples for {tool_name}")

print(f"\n✅ Total new records generated (all classes): {len(all_new_records)}")

In [ ]:
# ── Cell 5: Dedup new queries, assign tool_ids, merge, save before/after ─────
def normalize_q(q): return re.sub(r"[^\w\s]", "", str(q).lower()).strip()

existing_norms = set(normalize_q(r["query"]) for r in qwen_data)
seen_new_norms = set()
deduped_new = []
for r in all_new_records:
    n = normalize_q(r["query"])
    if n in existing_norms or n in seen_new_norms:
        continue
    seen_new_norms.add(n)
    deduped_new.append(r)

print(f"New records after dedup: {len(deduped_new)} (removed {len(all_new_records)-len(deduped_new)} duplicates)")

for r in deduped_new:
    r["tool_id"] = qwen_mapping["class_to_local_id"][r["tool"]]

qwen35_balanced = qwen_data + deduped_new
pd.DataFrame(qwen35_balanced).to_csv(OUT/"class_mappings"/"qwen35_balanced_final.csv", index=False)

after_counts = []
for tool_name in qwen_classes:
    rows = [r for r in qwen35_balanced if r["tool"]==tool_name]
    after_counts.append(dict(tool_name=tool_name, total_samples=len(rows),
                              unique_queries=len(set(r["query"] for r in rows)),
                              unique_images=len(set(r["img_id"] for r in rows))))
after_df = pd.DataFrame(after_counts).sort_values("total_samples")
after_df.to_csv(OUT/"class_mappings"/"qwen35_class_distribution_after.csv", index=False)

print(f"\n=== Before -> After for corrected classes ===")
for tool_name in STARVED_CLASSES:
    before_n = int(qwen_dist_df[qwen_dist_df.tool_name==tool_name].total_samples.iloc[0])
    after_n = int(after_df[after_df.tool_name==tool_name].total_samples.iloc[0])
    print(f"  {tool_name}: {before_n} -> {after_n}")

print(f"\nAny class still below 100: {(after_df['total_samples']<100).sum()}")

In [ ]:
# ── Verify what's actually on disk right now ──────────────────────────────────
print("Files saved so far:")
for f in OUT.rglob("*"):
    if f.is_file():
        print(f"  {f.relative_to(OUT)}  ({f.stat().st_size/1e3:.1f} KB)")

In [ ]:
# ── Cell 5b: Safety checkpoint — save intermediate generation results ────────
with open(OUT/"class_mappings"/"all_new_records_raw.json", "w") as f:
    json.dump(all_new_records, f, indent=2)
with open(OUT/"class_mappings"/"deduped_new_records.json", "w") as f:
    json.dump(deduped_new, f, indent=2)

print(f"✅ Checkpointed {len(all_new_records)} raw + {len(deduped_new)} deduped new records to disk")
print("Files now saved:")
for f in OUT.rglob("*"):
    if f.is_file():
        print(f"  {f.relative_to(OUT)}")

In [ ]:
# ── Cell 6: MiniLM + Pixel-CLIP embeddings for new samples only ──────────────
from sentence_transformers import SentenceTransformer

sbert = SentenceTransformer('all-MiniLM-L6-v2', device=str(DEVICE))
for p in sbert.parameters(): p.requires_grad = False

new_queries = [r["query"] for r in deduped_new]
new_text_emb = sbert.encode(new_queries, batch_size=256, show_progress_bar=True, normalize_embeddings=True)
print(f"✅ New text embeddings: {new_text_emb.shape}")

# Pixel-CLIP: reuse existing cached vectors where the img_id already has one
# (new samples reuse REAL existing COCO images from the corpus, so almost all
# should already be cached — only genuinely new img_ids need fresh encoding)
missing_clip_ids = [r["img_id"] for r in deduped_new if int(r["img_id"]) not in qwen_clip]
print(f"New samples needing fresh CLIP encoding: {len(missing_clip_ids)} / {len(deduped_new)}")

if missing_clip_ids:
    import open_clip
    from PIL import Image
    clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
        'ViT-B-32', pretrained='openai', force_quick_gelu=True
    )
    clip_model = clip_model.to(DEVICE).eval()

    def find_coco_image(img_id):
        for d in [KNOWN_IMG_DIR, NEW_IMG_DIR]:
            if d is None: continue
            p = Path(d) / f"{img_id}.jpg"
            if p.exists(): return p
        return None

    newly_encoded = 0
    for img_id in set(missing_clip_ids):
        img_path = find_coco_image(img_id)
        if img_path is None:
            continue
        image = Image.open(img_path).convert("RGB")
        pixel_values = clip_preprocess(image).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            feat = clip_model.encode_image(pixel_values)
            feat = (feat / feat.norm(dim=-1, keepdim=True)).cpu().numpy()[0]
        qwen_clip[int(img_id)] = feat
        newly_encoded += 1
    print(f"✅ Freshly encoded {newly_encoded} new images")

new_clip_vecs = np.stack([qwen_clip[int(r["img_id"])] for r in deduped_new])
X_new = np.concatenate([new_text_emb, new_clip_vecs], axis=1).astype(np.float32)
print(f"\n✅ New sample feature matrix: {X_new.shape}")

# Merge with existing canonical features
X_qwen_balanced = np.concatenate([X_qwen_896, X_new], axis=0)
y_qwen_balanced_local = np.concatenate([
    y_qwen_local,
    np.array([r["tool_id"] for r in deduped_new], dtype=np.int64)
])

print(f"\n✅ Combined balanced dataset: X={X_qwen_balanced.shape}  y={y_qwen_balanced_local.shape}")
assert X_qwen_balanced.shape[0] == y_qwen_balanced_local.shape[0] == len(qwen35_balanced)
print("✅ Row/label/feature alignment verified — no missing IDs")

np.save(OUT/"class_mappings"/"X_qwen_balanced.npy", X_qwen_balanced)
np.save(OUT/"class_mappings"/"y_qwen_balanced_local.npy", y_qwen_balanced_local)
print("✅ Saved balanced feature matrix and labels")

In [ ]:
import shutil

shutil.make_archive(
    "/kaggle/working/kaggle_working_backup",
    "zip",
    "/kaggle/working"
)

print("/kaggle/working/kaggle_working_backup.zip")

In [ ]:
# ── Cell 0: Locate datasets, output scaffold ──────────────────────────────────
from pathlib import Path
import json, shutil, os, sys, re
import numpy as np, torch, torch.nn as nn, pandas as pd
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.metrics import f1_score, confusion_matrix, classification_report

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
print(f"Device: {DEVICE}")

def find_dir(*fragments):
    for frag in fragments:
        m = [p for p in Path("/kaggle/input").rglob("*") if p.is_dir() and frag in p.name.lower()]
        if m: return m[0]
    return None

def find_file(*fragments):
    for frag in fragments:
        m = [p for p in Path("/kaggle/input").rglob("*") if p.is_file() and frag in p.name.lower()]
        if m: return m[0]
    return None

KAN_QWEN = find_dir("kan_qwen", "kan-qwen")
KAN_LLAMA = find_dir("kan_llama", "llama-kan")
PREV_V3 = find_dir("pre-run-v4-1")
PREV_V4_SHARED = find_dir("pre-run-v4") and [d for d in Path("/kaggle/input").rglob("*") if d.is_dir() and "pre-run-v4" in d.name.lower() and "v4-1" not in d.name.lower()]
PREV_V4_SHARED = PREV_V4_SHARED[0] if PREV_V4_SHARED else None
BACKUP = find_dir("kaggle_working_backup", "kaggle-working-backup")
KNOWN_IMG_DIR = find_dir("known")
NEW_IMG_DIR = find_dir("new")
REVIEW_B = find_file("review_b_ambiguous_queries")

for name, p in [("KAN_QWEN",KAN_QWEN),("KAN_LLAMA",KAN_LLAMA),("PREV_V3",PREV_V3),
                ("PREV_V4_SHARED",PREV_V4_SHARED),("BACKUP",BACKUP),
                ("KNOWN_IMG_DIR",KNOWN_IMG_DIR),("NEW_IMG_DIR",NEW_IMG_DIR),("REVIEW_B",REVIEW_B)]:
    print(f"{'✅' if p else '❌'} {name}: {p}")

OUT = Path("/kaggle/working/outputs")
for sub in ["class_mappings","splits","canonical","human_benchmark","sanity_checks",
            "leakage_audit","pixel_clip_provenance","validation","part4_eval_prep"]:
    (OUT/sub).mkdir(parents=True, exist_ok=True)
print("✅ scaffold ready")

if not Path("/kaggle/working/fast-kan").exists():
    os.system("git clone https://github.com/ZiyaoLi/fast-kan.git /kaggle/working/fast-kan")
sys.path.append("/kaggle/working/fast-kan")

In [ ]:
# ── Cell 1: Shared architecture + helpers ─────────────────────────────────────
from fastkan import FastKAN

class KANRoute(nn.Module):
    def __init__(self, input_dim=896, proj_dim=48, hidden_dims=[48], num_tools=35, num_grids=4):
        super().__init__()
        self.proj = nn.Linear(input_dim, proj_dim)
        self.norm = nn.LayerNorm(proj_dim)
        self.kan  = FastKAN(layers_hidden=[proj_dim]+hidden_dims+[num_tools], num_grids=num_grids)
    def forward(self, x): return self.kan(self.norm(self.proj(x.float())))
    def count_parameters(self): return sum(p.numel() for p in self.parameters() if p.requires_grad)

class MLPBaseline(nn.Module):
    def __init__(self, input_dim=896, proj_dim=96, hidden_dims=[64,32], num_tools=35, dropout=0.3):
        super().__init__()
        layers = [nn.Linear(input_dim, proj_dim), nn.LayerNorm(proj_dim), nn.GELU(), nn.Dropout(dropout)]
        prev = proj_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.GELU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, num_tools))
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x.float())
    def count_parameters(self): return sum(p.numel() for p in self.parameters() if p.requires_grad)

class ToolRoutingDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32); self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

def evaluate_full(model, loader):
    model.eval(); preds, labels = [], []
    with torch.no_grad():
        for X_b,y_b in loader:
            X_b,y_b = X_b.float().to(DEVICE), y_b.to(DEVICE)
            p = model(X_b).argmax(-1)
            preds.extend(p.cpu().numpy()); labels.extend(y_b.cpu().numpy())
    return np.array(preds), np.array(labels)

def train_model(model, tr, va, name, epochs=60, lr=1e-3, wd=1e-4, is_kan=False, save_dir=None):
    if is_kan: lr, epochs = 5e-4, 80
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, steps_per_epoch=len(tr), epochs=epochs, pct_start=0.1)
    crit = nn.CrossEntropyLoss(label_smoothing=0.05)
    best, path = 0.0, (save_dir or OUT)/f"{name}_best.pt"
    for ep in range(1, epochs+1):
        model.train()
        for X_b,y_b in tr:
            X_b,y_b = X_b.float().to(DEVICE), y_b.to(DEVICE)
            opt.zero_grad(); loss = crit(model(X_b), y_b); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sch.step()
        model.eval(); c,t=0,0
        with torch.no_grad():
            for X_b,y_b in va:
                X_b,y_b = X_b.float().to(DEVICE), y_b.to(DEVICE)
                c += (model(X_b).argmax(-1)==y_b).sum().item(); t += len(y_b)
        acc = c/t
        if acc > best: best = acc; torch.save(model.state_dict(), path)
        if ep%20==0 or ep==1: print(f"  [{name}] ep{ep}/{epochs} val={acc:.4f}")
    model.load_state_dict(torch.load(path))
    return best
print("✅ ready")

In [ ]:
# ── Diagnose: find where the backup actually landed ───────────────────────────
from pathlib import Path

for p in Path("/kaggle/input").iterdir():
    print(p)

print("\nSearching for X_qwen_balanced.npy anywhere under /kaggle/input:")
matches = list(Path("/kaggle/input").rglob("X_qwen_balanced.npy"))
print(matches)

In [ ]:
# ── Fix: point BACKUP at wherever X_qwen_balanced.npy actually is ────────────
found = list(Path("/kaggle/input").rglob("X_qwen_balanced.npy"))
if found:
    BACKUP = found[0].parent  # the class_mappings folder itself
    print(f"BACKUP set to: {BACKUP}")
else:
    print("❌ Still not found — the kaggle_working_backup.zip dataset may not be attached to this notebook.")
    print("   Go to Add Data and confirm it's actually added as an input to this session.")

In [ ]:
# ── Cell 2: Reload raw Qwen/Llama data, class mappings, and Item 1/2 outputs ─
with open(KAN_QWEN / "augmented_dataset_final.json") as f:
    qwen_data = json.load(f)
qwen_text = np.load(KAN_QWEN / "aug_text_embeddings.npy")
qc = np.load(KAN_QWEN / "clip_features.npz", allow_pickle=True)
qwen_clip = {int(k):v for k,v in qc[qc.files[0]].item().items()}
qwen_clip_vecs = np.stack([qwen_clip[int(r["img_id"])] for r in qwen_data])
X_qwen_896 = np.concatenate([qwen_text, qwen_clip_vecs], axis=1).astype(np.float32)

with open(KAN_LLAMA / "raw_dataset_FINAL.json") as f:
    llama_data = json.load(f)
llama_text = np.load(KAN_LLAMA / "text_embeddings.npy")
lc = np.load(KAN_LLAMA / "clip_features.npz", allow_pickle=True)
llama_clip = {int(k):v for k,v in lc[lc.files[0]].item().items()}
llama_clip_vecs = np.stack([llama_clip[int(r["img_id"])] for r in llama_data])
X_llama_896 = np.concatenate([llama_text, llama_clip_vecs], axis=1).astype(np.float32)

with open(next(PREV_V3.rglob("qwen_35_class_mapping.json"))) as f:
    qwen_mapping = json.load(f)
with open(next(PREV_V3.rglob("llama_35_class_mapping.json"))) as f:
    llama_mapping = json.load(f)
with open(next(PREV_V3.rglob("shared_29_class_mapping.json"))) as f:
    shared_mapping = json.load(f)

qwen_classes = sorted(qwen_mapping["class_to_local_id"].keys())
llama_classes = sorted(llama_mapping["class_to_local_id"].keys())
shared_classes = sorted(shared_mapping["class_to_local_id"].keys())

y_qwen_local = np.array([qwen_mapping["class_to_local_id"][r["tool"]] for r in qwen_data], dtype=np.int64)
y_llama_local = np.array([llama_mapping["class_to_local_id"][r["tool"]] for r in llama_data], dtype=np.int64)

print(f"Qwen: {X_qwen_896.shape} | Llama: {X_llama_896.shape} | Shared classes: {len(shared_classes)}")

# ── Reload Item 1/2 outputs from the backup (no need to regenerate!) ─────────
X_qwen_balanced = np.load(next(BACKUP.rglob("X_qwen_balanced.npy")))
y_qwen_balanced_local = np.load(next(BACKUP.rglob("y_qwen_balanced_local.npy")))

with open(next(BACKUP.rglob("deduped_new_records.json"))) as f:
    deduped_new = json.load(f)
with open(next(BACKUP.rglob("all_new_records_raw.json"))) as f:
    all_new_records = json.load(f)

qwen35_balanced_df = pd.read_csv(next(BACKUP.rglob("qwen35_balanced_final.csv")))
qwen_dist_before = pd.read_csv(next(BACKUP.rglob("qwen35_class_distribution_before.csv")))
qwen_dist_after = pd.read_csv(next(BACKUP.rglob("qwen35_class_distribution_after.csv")))

print(f"\n✅ Reloaded Item 1/2 outputs:")
print(f"   X_qwen_balanced: {X_qwen_balanced.shape}")
print(f"   y_qwen_balanced_local: {y_qwen_balanced_local.shape}")
print(f"   deduped_new records: {len(deduped_new)}")
print(f"   qwen35_balanced_final.csv: {len(qwen35_balanced_df)} rows")
assert X_qwen_balanced.shape[0] == y_qwen_balanced_local.shape[0] == len(qwen35_balanced_df)
print("✅ Alignment verified — no work lost, ready to continue from Item 3")

qwen35_balanced = qwen35_balanced_df.to_dict("records")
below_100 = qwen_dist_before[qwen_dist_before["total_samples"] < 100]
after_df = qwen_dist_after
STARVED_CLASSES = below_100["tool_name"].tolist()
print(f"\nOriginally starved classes (now fixed): {STARVED_CLASSES}")

In [ ]:
# ── Cell 3: Union-find grouping — same normalized query OR same (img_id, tool) ─
def normalize_q(q):
    return re.sub(r"[^\w\s]", "", str(q).lower()).strip()

class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))
    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x
    def union(self, x, y):
        rx, ry = self.find(x), self.find(y)
        if rx != ry: self.parent[rx] = ry

def build_leakage_safe_groups(records):
    n = len(records)
    uf = UnionFind(n)

    norm_to_idx = {}
    for i, r in enumerate(records):
        norm = normalize_q(r["query"])
        if norm in norm_to_idx:
            uf.union(i, norm_to_idx[norm])
        else:
            norm_to_idx[norm] = i

    imgtool_to_idx = {}
    for i, r in enumerate(records):
        key = (r["img_id"], r["tool"])
        if key in imgtool_to_idx:
            uf.union(i, imgtool_to_idx[key])
        else:
            imgtool_to_idx[key] = i

    groups = {}
    for i in range(n):
        root = uf.find(i)
        groups.setdefault(root, []).append(i)
    return list(groups.values())

qwen_groups = build_leakage_safe_groups(qwen35_balanced)
llama_groups = build_leakage_safe_groups(llama_data)
print(f"Qwen: {len(qwen35_balanced)} rows -> {len(qwen_groups)} leakage-safe groups")
print(f"Llama: {len(llama_data)} rows -> {len(llama_groups)} leakage-safe groups")
print(f"Largest Qwen group size: {max(len(g) for g in qwen_groups)}")
print(f"Largest Llama group size: {max(len(g) for g in llama_groups)}")

In [ ]:
# ── Reload the teacher model before regenerating ──────────────────────────────
from transformers import AutoTokenizer, AutoModelForCausalLM
import random

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
print(f"Loading {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
teacher = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="auto",
    low_cpu_mem_usage=True, trust_remote_code=True,
)
teacher.eval()
print(f"✅ Loaded. VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")

TOOL_DESCRIPTIONS = {
    "image_colorization": "add realistic color to a grayscale or black-and-white image",
    "matting": "extract a foreground subject with soft, fine-detailed edges (hair, fur, transparency)",
    "pose_estimation": "determine body posture and joint/skeleton positions of a person",
    "face_detection": "locate and identify human faces in an image",
    "facial_expression_recognition": "identify the emotional expression on a person's face",
    "point_cloud_generation": "generate 3D point-cloud data from an image",
    "text_detection_ocr": "detect and read visible text in an image",
    "surface_normal_estimation": "estimate the 3D orientation of surfaces in an image",
    "logo_detection": "identify brand logos visible in an image",
}

def build_targeted_prompt(objects, tool_name):
    obj_str = ", ".join(objects) if objects else "various objects"
    messages = [
        {"role": "system", "content": (
            "You are a precise JSON generator for an AI benchmark dataset. "
            "You ALWAYS output valid JSON and nothing else. "
            "No markdown, no explanation, no code blocks."
        )},
        {"role": "user", "content": (
            f"Image contains: {obj_str}\n\n"
            f"Tool: {tool_name} — {TOOL_DESCRIPTIONS[tool_name]}\n\n"
            "Write ONE realistic, natural user query that would require this exact tool. "
            "Vary the phrasing style each time (direct request, casual question, or descriptive request).\n\n"
            'Output ONLY this JSON: {"query": "..."}'
        )}
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

print("✅ tokenizer, teacher, TOOL_DESCRIPTIONS, build_targeted_prompt all ready")

In [ ]:
# ── Cell 4 (regeneration, fixed): force wide image diversity, no repeats ─────
def generate_for_class_diverse(tool_name, target_count, all_records, batch_size=8, max_new_tokens=100, min_unique_images=None):
    """Same generation logic, but SAMPLES WITHOUT REPLACEMENT across the full
    image pool, guaranteeing each generated sample uses a DIFFERENT image
    (up to the pool size), preventing large (img_id,tool) groups from forming."""
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    min_unique_images = min_unique_images or target_count  # default: 1 image per sample
    unique_img_pool = list({r["img_id"]: r.get("objects", []) for r in all_records}.items())
    random.seed(SEED)
    random.shuffle(unique_img_pool)

    if len(unique_img_pool) < min_unique_images:
        print(f"⚠️ Only {len(unique_img_pool)} unique images available, "
              f"wanted {min_unique_images} — will need to reuse some images")

    new_records = []
    pool_ptr = 0
    while len(new_records) < target_count and pool_ptr < len(unique_img_pool) * 2:
        batch = []
        for _ in range(batch_size):
            img_id, objs = unique_img_pool[pool_ptr % len(unique_img_pool)]
            batch.append((img_id, objs))
            pool_ptr += 1
        prompts = [build_targeted_prompt(objs, tool_name) for _, objs in batch]
        inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True, max_length=512).to(DEVICE)
        with torch.no_grad():
            output_ids = teacher.generate(
                **inputs, max_new_tokens=max_new_tokens, do_sample=True,
                temperature=0.8, top_p=0.9, repetition_penalty=1.15,
                pad_token_id=tokenizer.pad_token_id,
            )
        input_len = inputs["input_ids"].shape[1]
        for (img_id, objs), out_ids in zip(batch, output_ids):
            raw = tokenizer.decode(out_ids[input_len:], skip_special_tokens=True).strip()
            try:
                if "```" in raw:
                    raw = raw.split("```")[1].replace("json","",1).strip()
                parsed = json.loads(raw)
                query_text = parsed.get("query", "").strip()
                if query_text and len(new_records) < target_count:
                    new_records.append({
                        "img_id": img_id, "query": query_text, "tool": tool_name,
                        "objects": objs, "source": "item1_correction_v2_diverse",
                    })
            except Exception:
                continue
        if pool_ptr % 80 == 0:
            print(f"  [{tool_name}] {len(new_records)}/{target_count} generated, "
                  f"{len(set(r['img_id'] for r in new_records))} unique images used so far")
    return new_records

PROBLEM_CLASSES = ["face_detection", "facial_expression_recognition", "image_colorization",
                    "logo_detection", "matting"]

regenerated_records = []
for tool_name in PROBLEM_CLASSES:
    print(f"\nRegenerating '{tool_name}' with forced image diversity...")
    new = generate_for_class_diverse(tool_name, 150, qwen_data, min_unique_images=150)
    n_unique = len(set(r["img_id"] for r in new))
    print(f"  ✅ Generated {len(new)} samples across {n_unique} unique images (target: 150 unique)")
    regenerated_records.extend(new)

In [ ]:
# ── Cell 5: Verify ZERO leakage (Item 3's required checks) ───────────────────
def verify_zero_leakage(idx_tr, idx_va, idx_te, records, name):
    def get_norms(idxs): return set(normalize_q(records[i]["query"]) for i in idxs)
    def get_imgtool(idxs): return set((records[i]["img_id"], records[i]["tool"]) for i in idxs)

    norm_tr, norm_va, norm_te = get_norms(idx_tr), get_norms(idx_va), get_norms(idx_te)
    it_tr, it_va, it_te = get_imgtool(idx_tr), get_imgtool(idx_va), get_imgtool(idx_te)

    results = {
        "norm_train_test": len(norm_tr & norm_te),
        "norm_train_val": len(norm_tr & norm_va),
        "norm_val_test": len(norm_va & norm_te),
        "imgtool_train_test": len(it_tr & it_te),
        "imgtool_train_val": len(it_tr & it_va),
        "imgtool_val_test": len(it_va & it_te),
    }
    print(f"\n=== {name} leakage check ===")
    for k, v in results.items():
        print(f"  {'✅' if v==0 else '❌'} {k}: {v}")
    return results, all(v == 0 for v in results.values())

qwen_leak_results, qwen_leak_zero = verify_zero_leakage(idx_tr_Q, idx_va_Q, idx_te_Q, qwen35_balanced, "Qwen-35")
llama_leak_results, llama_leak_zero = verify_zero_leakage(idx_tr_L, idx_va_L, idx_te_L, llama_data, "Llama-35")

with open(OUT/"leakage_audit"/"split_leakage_audit_FINAL.md", "w") as f:
    f.write("# Split Leakage Audit — Final\n\n")
    f.write("Grouping rule: same normalized query OR same (image_id, tool) -> same split, via union-find.\n\n")
    for name, results, zero in [("Qwen-35", qwen_leak_results, qwen_leak_zero), ("Llama-35", llama_leak_results, llama_leak_zero)]:
        f.write(f"## {name}\n\n")
        for k, v in results.items():
            f.write(f"- {k}: {v}\n")
        f.write(f"\n**LEAKAGE FREE**: {zero}\n\n")

pd.DataFrame({"split":"train","idx":idx_tr_Q}).to_csv(OUT/"splits"/"qwen35_train_manifest.csv", index=False)
pd.DataFrame({"split":"val","idx":idx_va_Q}).to_csv(OUT/"splits"/"qwen35_val_manifest.csv", index=False)
pd.DataFrame({"split":"test","idx":idx_te_Q}).to_csv(OUT/"splits"/"qwen35_test_manifest.csv", index=False)
pd.DataFrame({"split":"train","idx":idx_tr_L}).to_csv(OUT/"splits"/"llama35_train_manifest.csv", index=False)
pd.DataFrame({"split":"val","idx":idx_va_L}).to_csv(OUT/"splits"/"llama35_val_manifest.csv", index=False)
pd.DataFrame({"split":"test","idx":idx_te_L}).to_csv(OUT/"splits"/"llama35_test_manifest.csv", index=False)

print(f"\n✅ Qwen leakage-free: {qwen_leak_zero}  |  Llama leakage-free: {llama_leak_zero}")

In [ ]:
# ── Cell 6: Retrain Qwen-35 KAN/MLP on balanced + leakage-safe splits ────────
X_train_Q = X_qwen_balanced[idx_tr_Q]; y_train_Q = y_qwen_balanced_local[idx_tr_Q]
X_val_Q = X_qwen_balanced[idx_va_Q]; y_val_Q = y_qwen_balanced_local[idx_va_Q]
X_test_Q = X_qwen_balanced[idx_te_Q]; y_test_Q = y_qwen_balanced_local[idx_te_Q]

train_loader_q = DataLoader(ToolRoutingDataset(X_train_Q, y_train_Q), batch_size=256, shuffle=True)
val_loader_q = DataLoader(ToolRoutingDataset(X_val_Q, y_val_Q), batch_size=256, shuffle=False)
test_loader_q = DataLoader(ToolRoutingDataset(X_test_Q, y_test_Q), batch_size=256, shuffle=False)

kan_qwen35_final = KANRoute(896, 48, [48], 35, 4).to(DEVICE)
mlp_qwen35_final = MLPBaseline(896, 96, [64,32], 35).to(DEVICE)

print("Training Qwen-35 KAN (corrected data)...")
train_model(kan_qwen35_final, train_loader_q, val_loader_q, "qwen35_kan_final", is_kan=True, save_dir=OUT/"canonical")
print("\nTraining Qwen-35 MLP (corrected data)...")
train_model(mlp_qwen35_final, train_loader_q, val_loader_q, "qwen35_mlp_final", is_kan=False, save_dir=OUT/"canonical")

kan_preds_q, kan_labels_q = evaluate_full(kan_qwen35_final, test_loader_q)
mlp_preds_q, mlp_labels_q = evaluate_full(mlp_qwen35_final, test_loader_q)

qwen_kan_f1 = f1_score(kan_labels_q, kan_preds_q, average="macro", zero_division=0)
qwen_mlp_f1 = f1_score(mlp_labels_q, mlp_preds_q, average="macro", zero_division=0)
print(f"\nQwen-35 KAN: Top-1={np.mean(kan_preds_q==kan_labels_q)*100:.2f}%  Macro F1={qwen_kan_f1:.4f}")
print(f"Qwen-35 MLP: Top-1={np.mean(mlp_preds_q==mlp_labels_q)*100:.2f}%  Macro F1={qwen_mlp_f1:.4f}")
print("\n(Do NOT expect these to match old numbers — leakage removal + rebalancing legitimately changes results)")

In [ ]:
# ── Cell 7: Per-class report, confusion matrix, macro-F1 class-coverage check ─
import matplotlib.pyplot as plt, seaborn as sns

local_id_to_name = {v: k for k, v in qwen_mapping["class_to_local_id"].items()}
present_classes = sorted(set(kan_labels_q.tolist()) | set(mlp_labels_q.tolist()))
class_names = [local_id_to_name[i] for i in present_classes]

for name, preds, labels in [("kan", kan_preds_q, kan_labels_q), ("mlp", mlp_preds_q, mlp_labels_q)]:
    report = classification_report(labels, preds, labels=list(range(35)), target_names=[local_id_to_name[i] for i in range(35)], output_dict=True, zero_division=0)
    pd.DataFrame(report).T.to_csv(OUT/"sanity_checks"/f"qwen35_{name}_per_class_results.csv")

cm = confusion_matrix(kan_labels_q, kan_preds_q, labels=list(range(35)))
plt.figure(figsize=(16,14))
sns.heatmap(cm, xticklabels=[local_id_to_name[i] for i in range(35)], yticklabels=[local_id_to_name[i] for i in range(35)], cmap="Blues")
plt.title("Qwen-35 KAN Confusion Matrix (corrected)")
plt.xticks(rotation=90); plt.yticks(rotation=0); plt.tight_layout()
plt.savefig(OUT/"sanity_checks"/"qwen35_confusion_matrix.png", dpi=120)
plt.close()

classes_in_test = len(set(y_test_Q.tolist()))
classes_in_macro_f1 = len(present_classes)
class_support = pd.Series(y_test_Q).value_counts()

MACRO_F1_CLASS_COVERAGE = f"{classes_in_macro_f1}/35"
print(f"Classes in test set: {classes_in_test}/35")
print(f"Classes used for macro-F1: {classes_in_macro_f1}/35")
print(f"Min class support: {class_support.min()}  Max: {class_support.max()}")
print(f"MACRO_F1_CLASS_COVERAGE = {MACRO_F1_CLASS_COVERAGE}")

macro_verified_v4 = (classes_in_test == 35)
if not macro_verified_v4:
    missing = set(range(35)) - set(y_test_Q.tolist())
    print(f"\n⚠️ Still missing from test: {[local_id_to_name[i] for i in missing]}")

results_summary = pd.DataFrame([
    dict(model="KAN", top1=np.mean(kan_preds_q==kan_labels_q), macro_f1=qwen_kan_f1, weighted_f1=f1_score(kan_labels_q,kan_preds_q,average='weighted',zero_division=0)),
    dict(model="MLP", top1=np.mean(mlp_preds_q==mlp_labels_q), macro_f1=qwen_mlp_f1, weighted_f1=f1_score(mlp_labels_q,mlp_preds_q,average='weighted',zero_division=0)),
])
results_summary.to_csv(OUT/"sanity_checks"/"qwen35_final_results.csv", index=False)
print(f"\n{results_summary.to_string(index=False)}")

In [ ]:
# ── Check exactly which class has only 1 test sample ──────────────────────────
test_class_counts = pd.Series(y_test_Q).value_counts().sort_values()
print("Lowest-support classes in Qwen test set:")
for local_id, count in test_class_counts.head(5).items():
    print(f"  {local_id_to_name[local_id]}: {count} test samples")

In [ ]:
# ── Diagnose which 3 Llama classes are still missing, and why ────────────────
llama_local_id_to_name = {v: k for k, v in llama_mapping["class_to_local_id"].items()}
missing_llama = set(range(35)) - set(y_llama_local[i] for i in idx_te_L)
print(f"Missing from Llama test set: {[llama_local_id_to_name[i] for i in missing_llama]}")

for local_id in missing_llama:
    name = llama_local_id_to_name[local_id]
    total_samples = sum(1 for r in llama_data if r["tool"] == name)
    # find which group(s) contain this class
    group_sizes_for_class = [len(g) for g in llama_groups if any(llama_data[i]["tool"]==name for i in g)]
    print(f"  {name}: {total_samples} total samples, contained in group(s) of size {group_sizes_for_class}")

In [ ]:
# ── Cell: Splice diverse regenerated samples in place of the old narrow ones ─
PROBLEM_CLASSES = ["face_detection", "facial_expression_recognition", "image_colorization",
                    "logo_detection", "matting"]

# Remove OLD synthetic samples for these 5 classes from deduped_new
# (keep original qwen_data's tiny original counts for these classes untouched —
# those are the true "canonical" originals; we're only replacing the Item-1
# correction batch, not the pre-existing handful of real samples)
old_correction_removed = [
    r for r in deduped_new
    if not (r["tool"] in PROBLEM_CLASSES and r.get("source", "").startswith("item1_correction"))
]
print(f"Removed {len(deduped_new) - len(old_correction_removed)} old narrow-diversity samples")

# Dedup the new diverse batch against everything else (existing + already-kept new)
def normalize_q(q): return re.sub(r"[^\w\s]", "", str(q).lower()).strip()
existing_norms = set(normalize_q(r["query"]) for r in qwen_data) | set(normalize_q(r["query"]) for r in old_correction_removed)

seen_norms = set()
deduped_regenerated = []
for r in regenerated_records:
    n = normalize_q(r["query"])
    if n in existing_norms or n in seen_norms:
        continue
    seen_norms.add(n)
    deduped_regenerated.append(r)

print(f"New diverse samples after dedup: {len(deduped_regenerated)} (removed {len(regenerated_records)-len(deduped_regenerated)})")

# Assign tool_ids
for r in deduped_regenerated:
    r["tool_id"] = qwen_mapping["class_to_local_id"][r["tool"]]

# Final deduped_new = kept old corrections (for the other 4 classes) + new diverse ones
deduped_new = old_correction_removed + deduped_regenerated
print(f"\n✅ Final deduped_new: {len(deduped_new)} total correction records")

# Rebuild qwen35_balanced from scratch
qwen35_balanced = qwen_data + deduped_new
pd.DataFrame(qwen35_balanced).to_csv(OUT/"class_mappings"/"qwen35_balanced_final.csv", index=False)

# Recompute distribution-after
after_counts = []
for tool_name in qwen_classes:
    rows = [r for r in qwen35_balanced if r["tool"]==tool_name]
    after_counts.append(dict(tool_name=tool_name, total_samples=len(rows),
                              unique_queries=len(set(r["query"] for r in rows)),
                              unique_images=len(set(r["img_id"] for r in rows))))
after_df = pd.DataFrame(after_counts).sort_values("total_samples")
after_df.to_csv(OUT/"class_mappings"/"qwen35_class_distribution_after.csv", index=False)
print(f"\nAny class still below 100: {(after_df['total_samples']<100).sum()}")
print(after_df[after_df.tool_name.isin(PROBLEM_CLASSES)].to_string(index=False))

In [ ]:
# ── Cell: Embeddings for the newly-swapped-in diverse samples ────────────────
from sentence_transformers import SentenceTransformer

sbert = SentenceTransformer('all-MiniLM-L6-v2', device=str(DEVICE))
for p in sbert.parameters(): p.requires_grad = False

new_queries = [r["query"] for r in deduped_new]
new_text_emb = sbert.encode(new_queries, batch_size=256, show_progress_bar=True, normalize_embeddings=True)
print(f"✅ Text embeddings: {new_text_emb.shape}")

missing_clip_ids = [r["img_id"] for r in deduped_new if int(r["img_id"]) not in qwen_clip]
print(f"Samples needing fresh CLIP encoding: {len(missing_clip_ids)} / {len(deduped_new)}")

if missing_clip_ids:
    import open_clip
    from PIL import Image
    clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
        'ViT-B-32', pretrained='openai', force_quick_gelu=True
    )
    clip_model = clip_model.to(DEVICE).eval()

    def find_coco_image(img_id):
        for d in [KNOWN_IMG_DIR, NEW_IMG_DIR]:
            if d is None: continue
            p = Path(d) / f"{img_id}.jpg"
            if p.exists(): return p
        return None

    newly_encoded = 0
    for img_id in set(missing_clip_ids):
        img_path = find_coco_image(img_id)
        if img_path is None:
            continue
        image = Image.open(img_path).convert("RGB")
        pixel_values = clip_preprocess(image).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            feat = clip_model.encode_image(pixel_values)
            feat = (feat / feat.norm(dim=-1, keepdim=True)).cpu().numpy()[0]
        qwen_clip[int(img_id)] = feat
        newly_encoded += 1
    print(f"✅ Freshly encoded {newly_encoded} new images")

new_clip_vecs = np.stack([qwen_clip[int(r["img_id"])] for r in deduped_new])
X_new = np.concatenate([new_text_emb, new_clip_vecs], axis=1).astype(np.float32)

X_qwen_balanced = np.concatenate([X_qwen_896, X_new], axis=0)
y_qwen_balanced_local = np.concatenate([
    y_qwen_local,
    np.array([r["tool_id"] for r in deduped_new], dtype=np.int64)
])

print(f"\n✅ Rebuilt balanced dataset: X={X_qwen_balanced.shape}  y={y_qwen_balanced_local.shape}")
assert X_qwen_balanced.shape[0] == y_qwen_balanced_local.shape[0] == len(qwen35_balanced)

np.save(OUT/"class_mappings"/"X_qwen_balanced.npy", X_qwen_balanced)
np.save(OUT/"class_mappings"/"y_qwen_balanced_local.npy", y_qwen_balanced_local)
print("✅ Saved updated feature matrix and labels")

In [ ]:
# ── Cell: Fresh union-find grouping on updated qwen35_balanced ───────────────
def normalize_q(q):
    return re.sub(r"[^\w\s]", "", str(q).lower()).strip()

class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))
    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x
    def union(self, x, y):
        rx, ry = self.find(x), self.find(y)
        if rx != ry: self.parent[rx] = ry

def build_leakage_safe_groups(records):
    n = len(records)
    uf = UnionFind(n)
    norm_to_idx = {}
    for i, r in enumerate(records):
        norm = normalize_q(r["query"])
        if norm in norm_to_idx:
            uf.union(i, norm_to_idx[norm])
        else:
            norm_to_idx[norm] = i
    imgtool_to_idx = {}
    for i, r in enumerate(records):
        key = (r["img_id"], r["tool"])
        if key in imgtool_to_idx:
            uf.union(i, imgtool_to_idx[key])
        else:
            imgtool_to_idx[key] = i
    groups = {}
    for i in range(n):
        root = uf.find(i)
        groups.setdefault(root, []).append(i)
    return list(groups.values())

qwen_groups = build_leakage_safe_groups(qwen35_balanced)
print(f"Qwen: {len(qwen35_balanced)} rows -> {len(qwen_groups)} leakage-safe groups")
print(f"Largest group size: {max(len(g) for g in qwen_groups)}")

group_sizes_for_problem_classes = {}
for tool_name in PROBLEM_CLASSES:
    sizes = [len(g) for g in qwen_groups if any(qwen35_balanced[i]["tool"]==tool_name for i in g)]
    group_sizes_for_problem_classes[tool_name] = sorted(sizes, reverse=True)[:5]
    print(f"{tool_name}: top 5 group sizes = {group_sizes_for_problem_classes[tool_name]}")

In [ ]:
# ── Cell: Class-aware split (v2, the corrected version) ──────────────────────
def group_based_split_class_aware_v2(groups, y_local, seed=SEED, train_frac=0.80, val_frac=0.10):
    rng = np.random.RandomState(seed)
    group_sizes = sorted(range(len(groups)), key=lambda gi: len(groups[gi]))
    used_idx = set()
    val_idx, test_idx = [], []
    all_classes = set(y_local.tolist())

    def assign_smallest_available_group_for_class(cls, exclude_used):
        candidates = [(gi, len(groups[gi])) for gi in group_sizes
                      if not any(i in exclude_used for i in groups[gi])
                      and any(y_local[i] == cls for i in groups[gi])]
        return min(candidates, key=lambda x: x[1])[0] if candidates else None

    for cls in all_classes:
        if any(y_local[i]==cls for i in val_idx): continue
        gi = assign_smallest_available_group_for_class(cls, used_idx)
        if gi is not None:
            val_idx.extend(groups[gi]); used_idx.update(groups[gi])
        else:
            print(f"⚠️ Could not find an available group for class {cls} in val pass")

    for cls in all_classes:
        if any(y_local[i]==cls for i in test_idx): continue
        gi = assign_smallest_available_group_for_class(cls, used_idx)
        if gi is not None:
            test_idx.extend(groups[gi]); used_idx.update(groups[gi])
        else:
            print(f"⚠️ Could not find an available group for class {cls} in test pass")

    remaining = [gi for gi in group_sizes if not any(i in used_idx for i in groups[gi])]
    rng.shuffle(remaining)
    n_total = len(y_local)
    n_train_target = int(train_frac * n_total)

    train_idx = []
    for gi in remaining:
        if len(train_idx) < n_train_target:
            train_idx.extend(groups[gi])
        else:
            val_idx.extend(groups[gi])

    return train_idx, val_idx, test_idx

idx_tr_Q, idx_va_Q, idx_te_Q = group_based_split_class_aware_v2(qwen_groups, y_qwen_balanced_local)
print(f"\nQwen split: train={len(idx_tr_Q)} val={len(idx_va_Q)} test={len(idx_te_Q)}")

qwen_test_classes = len(set(y_qwen_balanced_local[i] for i in idx_te_Q))
print(f"Qwen test set classes present: {qwen_test_classes}/35")

test_class_counts = pd.Series(y_qwen_balanced_local[idx_te_Q]).value_counts().sort_values()
print(f"\nLowest-support classes in NEW test set:")
for local_id, count in test_class_counts.head(8).items():
    print(f"  {local_id_to_name[local_id]}: {count} test samples")

In [ ]:
# ── Fixed split function: remaining groups fill train -> val -> test properly ─
def group_based_split_class_aware_v3(groups, y_local, seed=SEED, train_frac=0.80, val_frac=0.10):
    rng = np.random.RandomState(seed)
    group_sizes = sorted(range(len(groups)), key=lambda gi: len(groups[gi]))
    used_idx = set()
    val_idx, test_idx = [], []
    all_classes = set(y_local.tolist())

    def assign_smallest_available_group_for_class(cls, exclude_used):
        candidates = [(gi, len(groups[gi])) for gi in group_sizes
                      if not any(i in exclude_used for i in groups[gi])
                      and any(y_local[i] == cls for i in groups[gi])]
        return min(candidates, key=lambda x: x[1])[0] if candidates else None

    # Guarantee pass: smallest group per class into val, then test
    for cls in all_classes:
        if any(y_local[i]==cls for i in val_idx): continue
        gi = assign_smallest_available_group_for_class(cls, used_idx)
        if gi is not None:
            val_idx.extend(groups[gi]); used_idx.update(groups[gi])

    for cls in all_classes:
        if any(y_local[i]==cls for i in test_idx): continue
        gi = assign_smallest_available_group_for_class(cls, used_idx)
        if gi is not None:
            test_idx.extend(groups[gi]); used_idx.update(groups[gi])

    # Fill REMAINING groups into train first, then top up val/test to their real targets
    remaining = [gi for gi in group_sizes if not any(i in used_idx for i in groups[gi])]
    rng.shuffle(remaining)

    n_total = len(y_local)
    n_train_target = int(train_frac * n_total)
    n_val_target = int(val_frac * n_total)
    # test_target is implicitly whatever's left (~10%)

    train_idx = []
    for gi in remaining:
        idxs = groups[gi]
        if len(train_idx) < n_train_target:
            train_idx.extend(idxs)
        elif len(val_idx) < n_val_target:
            val_idx.extend(idxs)
        else:
            test_idx.extend(idxs)  # <-- THE FIX: test actually gets filled now

    return train_idx, val_idx, test_idx

idx_tr_Q, idx_va_Q, idx_te_Q = group_based_split_class_aware_v3(qwen_groups, y_qwen_balanced_local)
print(f"Qwen split: train={len(idx_tr_Q)} val={len(idx_va_Q)} test={len(idx_te_Q)}")

qwen_test_classes = len(set(y_qwen_balanced_local[i] for i in idx_te_Q))
print(f"Qwen test set classes present: {qwen_test_classes}/35")

test_class_counts = pd.Series(y_qwen_balanced_local[idx_te_Q]).value_counts().sort_values()
print(f"\nLowest-support classes in test set now:")
for local_id, count in test_class_counts.head(10).items():
    print(f"  {local_id_to_name[local_id]}: {count} test samples")

In [ ]:
# ── Verify leakage-free on the corrected split ────────────────────────────────
def verify_zero_leakage(idx_tr, idx_va, idx_te, records, name):
    def get_norms(idxs): return set(normalize_q(records[i]["query"]) for i in idxs)
    def get_imgtool(idxs): return set((records[i]["img_id"], records[i]["tool"]) for i in idxs)
    norm_tr, norm_va, norm_te = get_norms(idx_tr), get_norms(idx_va), get_norms(idx_te)
    it_tr, it_va, it_te = get_imgtool(idx_tr), get_imgtool(idx_va), get_imgtool(idx_te)
    results = {
        "norm_train_test": len(norm_tr & norm_te), "norm_train_val": len(norm_tr & norm_va), "norm_val_test": len(norm_va & norm_te),
        "imgtool_train_test": len(it_tr & it_te), "imgtool_train_val": len(it_tr & it_va), "imgtool_val_test": len(it_va & it_te),
    }
    print(f"\n=== {name} leakage check ===")
    for k, v in results.items():
        print(f"  {'✅' if v==0 else '❌'} {k}: {v}")
    return results, all(v == 0 for v in results.values())

qwen_leak_results, qwen_leak_zero = verify_zero_leakage(idx_tr_Q, idx_va_Q, idx_te_Q, qwen35_balanced, "Qwen-35 (corrected)")

In [ ]:
# ── Retrain Qwen-35 KAN/MLP on the fully corrected data + splits ─────────────
X_train_Q = X_qwen_balanced[idx_tr_Q]; y_train_Q = y_qwen_balanced_local[idx_tr_Q]
X_val_Q = X_qwen_balanced[idx_va_Q]; y_val_Q = y_qwen_balanced_local[idx_va_Q]
X_test_Q = X_qwen_balanced[idx_te_Q]; y_test_Q = y_qwen_balanced_local[idx_te_Q]

train_loader_q = DataLoader(ToolRoutingDataset(X_train_Q, y_train_Q), batch_size=256, shuffle=True)
val_loader_q = DataLoader(ToolRoutingDataset(X_val_Q, y_val_Q), batch_size=256, shuffle=False)
test_loader_q = DataLoader(ToolRoutingDataset(X_test_Q, y_test_Q), batch_size=256, shuffle=False)

kan_qwen35_final = KANRoute(896, 48, [48], 35, 4).to(DEVICE)
mlp_qwen35_final = MLPBaseline(896, 96, [64,32], 35).to(DEVICE)

print("Training Qwen-35 KAN (fully corrected)...")
train_model(kan_qwen35_final, train_loader_q, val_loader_q, "qwen35_kan_final", is_kan=True, save_dir=OUT/"canonical")
print("\nTraining Qwen-35 MLP (fully corrected)...")
train_model(mlp_qwen35_final, train_loader_q, val_loader_q, "qwen35_mlp_final", is_kan=False, save_dir=OUT/"canonical")

kan_preds_q, kan_labels_q = evaluate_full(kan_qwen35_final, test_loader_q)
mlp_preds_q, mlp_labels_q = evaluate_full(mlp_qwen35_final, test_loader_q)

qwen_kan_f1 = f1_score(kan_labels_q, kan_preds_q, average="macro", zero_division=0)
qwen_mlp_f1 = f1_score(mlp_labels_q, mlp_preds_q, average="macro", zero_division=0)
print(f"\nQwen-35 KAN: Top-1={np.mean(kan_preds_q==kan_labels_q)*100:.2f}%  Macro F1={qwen_kan_f1:.4f}")
print(f"Qwen-35 MLP: Top-1={np.mean(mlp_preds_q==mlp_labels_q)*100:.2f}%  Macro F1={qwen_mlp_f1:.4f}")

classes_in_test = len(set(y_test_Q.tolist()))
print(f"\nClasses in test set: {classes_in_test}/35")
print(f"MACRO_F1_CLASS_COVERAGE = {classes_in_test}/35")
print(f"Min class support: {pd.Series(y_test_Q).value_counts().min()}  Max: {pd.Series(y_test_Q).value_counts().max()}")

In [ ]:
# ── Per-class report + confusion matrix (Item 4's required deliverables) ─────
import matplotlib.pyplot as plt, seaborn as sns

for name, preds, labels in [("kan", kan_preds_q, kan_labels_q), ("mlp", mlp_preds_q, mlp_labels_q)]:
    report = classification_report(labels, preds, labels=list(range(35)),
                                    target_names=[local_id_to_name[i] for i in range(35)],
                                    output_dict=True, zero_division=0)
    pd.DataFrame(report).T.to_csv(OUT/"sanity_checks"/f"qwen35_{name}_per_class_results.csv")

cm = confusion_matrix(kan_labels_q, kan_preds_q, labels=list(range(35)))
plt.figure(figsize=(16,14))
sns.heatmap(cm, xticklabels=[local_id_to_name[i] for i in range(35)],
            yticklabels=[local_id_to_name[i] for i in range(35)], cmap="Blues")
plt.title("Qwen-35 KAN Confusion Matrix (leakage-free, rebalanced)")
plt.xticks(rotation=90); plt.yticks(rotation=0); plt.tight_layout()
plt.savefig(OUT/"sanity_checks"/"qwen35_confusion_matrix.png", dpi=120)
plt.close()

results_summary = pd.DataFrame([
    dict(model="KAN", top1=np.mean(kan_preds_q==kan_labels_q), macro_f1=qwen_kan_f1,
         weighted_f1=f1_score(kan_labels_q,kan_preds_q,average='weighted',zero_division=0)),
    dict(model="MLP", top1=np.mean(mlp_preds_q==mlp_labels_q), macro_f1=qwen_mlp_f1,
         weighted_f1=f1_score(mlp_labels_q,mlp_preds_q,average='weighted',zero_division=0)),
])
results_summary.to_csv(OUT/"sanity_checks"/"qwen35_final_results.csv", index=False)
print("✅ Qwen-35 canonical artifacts saved: per-class reports, confusion matrix, final results CSV")
print(results_summary.to_string(index=False))

In [ ]:
# ── Llama: rebuild groups + corrected split (same v3 function) ───────────────
llama_groups = build_leakage_safe_groups(llama_data)
print(f"Llama: {len(llama_data)} rows -> {len(llama_groups)} leakage-safe groups")

idx_tr_L, idx_va_L, idx_te_L = group_based_split_class_aware_v3(llama_groups, y_llama_local)
print(f"Llama split: train={len(idx_tr_L)} val={len(idx_va_L)} test={len(idx_te_L)}")

llama_test_classes = len(set(y_llama_local[i] for i in idx_te_L))
print(f"Llama test set classes present: {llama_test_classes}/35")

llama_leak_results, llama_leak_zero = verify_zero_leakage(idx_tr_L, idx_va_L, idx_te_L, llama_data, "Llama-35 (v3 split)")

In [ ]:
# ── Cell: Llama-35 leakage-safe groups + corrected split ─────────────────────
llama_groups = build_leakage_safe_groups(llama_data)
print(f"Llama: {len(llama_data)} rows -> {len(llama_groups)} leakage-safe groups")
print(f"Largest group size: {max(len(g) for g in llama_groups)}")

idx_tr_L, idx_va_L, idx_te_L = group_based_split_class_aware_v3(llama_groups, y_llama_local)
print(f"\nLlama split: train={len(idx_tr_L)} val={len(idx_va_L)} test={len(idx_te_L)}")

llama_local_id_to_name = {v: k for k, v in llama_mapping["class_to_local_id"].items()}
llama_test_classes = len(set(y_llama_local[i] for i in idx_te_L))
print(f"Llama test set classes present: {llama_test_classes}/35")

missing_llama = set(range(35)) - set(y_llama_local[i] for i in idx_te_L)
if missing_llama:
    print(f"\nMissing classes: {[llama_local_id_to_name[i] for i in missing_llama]}")
    for local_id in missing_llama:
        name = llama_local_id_to_name[local_id]
        total = sum(1 for r in llama_data if r["tool"] == name)
        print(f"  {name}: {total} total samples in entire dataset")

test_class_counts_l = pd.Series(y_llama_local[idx_te_L]).value_counts().sort_values()
print(f"\nLowest-support classes in Llama test set:")
for local_id, count in test_class_counts_l.head(8).items():
    print(f"  {llama_local_id_to_name[local_id]}: {count} test samples")

In [ ]:
# ── Cell: Verify Llama-35 leakage-free ────────────────────────────────────────
llama_leak_results, llama_leak_zero = verify_zero_leakage(idx_tr_L, idx_va_L, idx_te_L, llama_data, "Llama-35 (corrected)")

pd.DataFrame({"split":"train","idx":idx_tr_L}).to_csv(OUT/"splits"/"llama35_train_manifest.csv", index=False)
pd.DataFrame({"split":"val","idx":idx_va_L}).to_csv(OUT/"splits"/"llama35_val_manifest.csv", index=False)
pd.DataFrame({"split":"test","idx":idx_te_L}).to_csv(OUT/"splits"/"llama35_test_manifest.csv", index=False)

In [ ]:
# ── Cell: Retrain Llama-35 KAN/MLP on corrected leakage-safe splits ──────────
X_train_L = X_llama_896[idx_tr_L]; y_train_L = y_llama_local[idx_tr_L]
X_val_L = X_llama_896[idx_va_L]; y_val_L = y_llama_local[idx_va_L]
X_test_L = X_llama_896[idx_te_L]; y_test_L = y_llama_local[idx_te_L]

train_loader_l = DataLoader(ToolRoutingDataset(X_train_L, y_train_L), batch_size=256, shuffle=True)
val_loader_l = DataLoader(ToolRoutingDataset(X_val_L, y_val_L), batch_size=256, shuffle=False)
test_loader_l = DataLoader(ToolRoutingDataset(X_test_L, y_test_L), batch_size=256, shuffle=False)

kan_llama35_final = KANRoute(896, 48, [48], 35, 4).to(DEVICE)
mlp_llama35_final = MLPBaseline(896, 96, [64,32], 35).to(DEVICE)

print("Training Llama-35 KAN (leakage-safe)...")
train_model(kan_llama35_final, train_loader_l, val_loader_l, "llama35_kan_final", is_kan=True, save_dir=OUT/"canonical")
print("\nTraining Llama-35 MLP (leakage-safe)...")
train_model(mlp_llama35_final, train_loader_l, val_loader_l, "llama35_mlp_final", is_kan=False, save_dir=OUT/"canonical")

kan_preds_l, kan_labels_l = evaluate_full(kan_llama35_final, test_loader_l)
mlp_preds_l, mlp_labels_l = evaluate_full(mlp_llama35_final, test_loader_l)

llama_kan_f1 = f1_score(kan_labels_l, kan_preds_l, average="macro", zero_division=0)
llama_mlp_f1 = f1_score(mlp_labels_l, mlp_preds_l, average="macro", zero_division=0)
print(f"\nLlama-35 KAN: Top-1={np.mean(kan_preds_l==kan_labels_l)*100:.2f}%  Macro F1={llama_kan_f1:.4f}")
print(f"Llama-35 MLP: Top-1={np.mean(mlp_preds_l==mlp_labels_l)*100:.2f}%  Macro F1={llama_mlp_f1:.4f}")
print(f"\nClasses in test set: {len(set(y_test_L.tolist()))}/35")

In [ ]:
# ── Cell: Llama-35 canonical artifacts + documented class-coverage caveat ────
for name, preds, labels in [("kan", kan_preds_l, kan_labels_l), ("mlp", mlp_preds_l, mlp_labels_l)]:
    report = classification_report(labels, preds, labels=list(range(35)),
                                    target_names=[llama_local_id_to_name[i] for i in range(35)],
                                    output_dict=True, zero_division=0)
    pd.DataFrame(report).T.to_csv(OUT/"sanity_checks"/f"llama35_{name}_per_class_results.csv")

cm_l = confusion_matrix(kan_labels_l, kan_preds_l, labels=list(range(35)))
plt.figure(figsize=(16,14))
sns.heatmap(cm_l, xticklabels=[llama_local_id_to_name[i] for i in range(35)],
            yticklabels=[llama_local_id_to_name[i] for i in range(35)], cmap="Blues")
plt.title("Llama-35 KAN Confusion Matrix (leakage-free)")
plt.xticks(rotation=90); plt.yticks(rotation=0); plt.tight_layout()
plt.savefig(OUT/"sanity_checks"/"llama35_confusion_matrix.png", dpi=120)
plt.close()

llama_results_summary = pd.DataFrame([
    dict(model="KAN", top1=np.mean(kan_preds_l==kan_labels_l), macro_f1=llama_kan_f1,
         weighted_f1=f1_score(kan_labels_l,kan_preds_l,average='weighted',zero_division=0)),
    dict(model="MLP", top1=np.mean(mlp_preds_l==mlp_labels_l), macro_f1=llama_mlp_f1,
         weighted_f1=f1_score(mlp_labels_l,mlp_preds_l,average='weighted',zero_division=0)),
])
llama_results_summary.to_csv(OUT/"sanity_checks"/"llama35_final_results.csv", index=False)

with open(OUT/"leakage_audit"/"split_leakage_audit_FINAL.md", "a") as f:
    f.write(f"\n\n## Llama-35 Final Class Coverage\n\n")
    f.write(f"Classes present in leakage-safe test set: {llama_test_classes}/35\n\n")
    if missing_llama:
        f.write("Missing (accepted limitation): " + ", ".join(llama_local_id_to_name[i] for i in missing_llama) + "\n\n")
        f.write("`video_object_tracking` and `temporal_action_localization` require video/temporal "
                "input incompatible with a static-image dataset (structural limitation, previously "
                "documented). `matting` has only 1 total sample in the entire 9,903-row Llama corpus, "
                "making it mathematically impossible to include in both train and test under any "
                "leakage-safe split.\n")

print("✅ Llama-35 canonical artifacts saved")
print(llama_results_summary.to_string(index=False))

In [ ]:
# ── Cell: Update CANONICAL_RESULTS.md, save artifact registry, zip everything ─
canonical_md = f"""# CANONICAL_RESULTS.md — v4 Final Blocker Fix

## LEGACY / SUPERSEDED
All v3 35-way checkpoints and results are superseded by the leakage-safe,
rebalanced retraining below.

## Canonical Models (v4)

| Model | Output size | Checkpoint | Top-1 | Macro F1 | Classes in test |
|---|---|---|---|---|---|
| Qwen-35 KAN | 35 | qwen35_kan_final_best.pt | {np.mean(kan_preds_q==kan_labels_q)*100:.2f}% | {qwen_kan_f1:.4f} | 35/35 |
| Qwen-35 MLP | 35 | qwen35_mlp_final_best.pt | {np.mean(mlp_preds_q==mlp_labels_q)*100:.2f}% | {qwen_mlp_f1:.4f} | 35/35 |
| Llama-35 KAN | 35 | llama35_kan_final_best.pt | {np.mean(kan_preds_l==kan_labels_l)*100:.2f}% | {llama_kan_f1:.4f} | {llama_test_classes}/35 |
| Llama-35 MLP | 35 | llama35_mlp_final_best.pt | {np.mean(mlp_preds_l==mlp_labels_l)*100:.2f}% | {llama_mlp_f1:.4f} | {llama_test_classes}/35 |

**Split methodology**: union-find connected-component grouping (same normalized
query text OR same (image_id, tool) -> same split), 80/10/10, class-aware
smallest-group-first assignment guaranteeing all reachable classes appear in
val and test.

**Qwen correction**: 1,233 new synthetic samples generated via Qwen2.5-7B-Instruct
(same model/method as original pipeline) for 9 originally-starved classes,
with forced image diversity (1 unique image per generated sample) to prevent
large leakage-groups from forming.

**Seed**: {SEED}
"""
with open(OUT/"canonical"/"CANONICAL_RESULTS.md", "w") as f:
    f.write(canonical_md)
print("✅ CANONICAL_RESULTS.md updated")

import zipfile
from datetime import datetime
ts = datetime.now().strftime("%Y%m%d_%H%M")
zip_path = Path(f"/kaggle/working/v4_blocker_fix_FINAL_{ts}.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in OUT.rglob("*"):
        if f.is_file():
            zf.write(f, f.relative_to(OUT.parent))
print(f"✅ {zip_path.name} ({zip_path.stat().st_size/1e6:.1f} MB)")